# Willett 2023 · speech T12

## Prepare the data

Run the commands below from the repository root with `bci-repro` active. See the [reader guide](../../docs/20261124_repository_migration/reader_workflow.md) for environment and English pronunciation resource setup.

### 1. Download

Download `competitionData.tar.gz` from [Dryad](https://doi.org/10.5061/dryad.x69p8czpq). Choose the September 1, 2023 release. Create `studies/20230831_willett/data/raw/dryad_x69p8czpq/` and save the download as `studies/20230831_willett/data/raw/dryad_x69p8czpq/dryad_x69p8czpq__competitionData.tar.gz`.

### 2. Extract

Create `studies/20230831_willett/data/extracted/dryad_x69p8czpq/` before running the extraction command.

```text
tar -xzf "studies/20230831_willett/data/raw/dryad_x69p8czpq/dryad_x69p8czpq__competitionData.tar.gz" -C "studies/20230831_willett/data/extracted/dryad_x69p8czpq"
```

After extraction, check for `studies/20230831_willett/data/extracted/dryad_x69p8czpq/competitionData/test/`. The same `competitionData/` directory also contains `train/` and `competitionHoldOut/`.

### 3. Prepare the model input

Convert the labeled training and evaluation `.mat` files to the HDF5 layout used by the trainer.

```text
python shared/tools/setup_upstream.py --study 20230831_willett
python studies/20230831_willett/code/prepare_data.py
```

**Output** `studies/20230831_willett/data/derived/dryad_x69p8czpq/hdf5/trainer_input/`. Each session has `data_train.hdf5` and/or `data_val.hdf5`. The session index is `studies/20230831_willett/data/derived/dryad_x69p8czpq/hdf5/dryad_x69p8czpq__session_description.csv`.

### 4. Export the common session file

Write one recording day to the session format shared by every study in this repository. The file stores each trial with the features as released and records the publisher file and position the trial came from. This example exports May 19.

```text
python studies/20230831_willett/code/export_sessions.py --session t12.2022.05.19
```

**Output** `studies/20230831_willett/data/processed/t12/dryad_x69p8czpq_2022-05-19.h5`. Add `--list` to see every session and whether it is exported, or use `--all` to export them all.

Check the file structure and trace every trial back to its `.mat` file with one command.

```text
python shared/tools/validate_session.py studies/20230831_willett/data/processed/t12/dryad_x69p8czpq_2022-05-19.h5 --source trials
```

**Output** A `PASS` line for the file, ending with `source trace: 200/200 trials re-read and identical`.


In [1]:
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "shared"))
from session_format import read_session

session_file = root / "studies/20230831_willett/data/processed/t12/dryad_x69p8czpq_2022-05-19.h5"
if session_file.is_file():
    with read_session(session_file) as session:
        print(f"{session.attrs['session']}: {len(session)} trials")
        trial = session.select(source_partition="test")[0]
        print(trial.trial_id, "·", trial.sentence)
        for name, array in trial.neural().items():
            print(f"  neural/{name}: {array.shape} {array.dtype}")
        labels = session.label_sets["phonemes"]
        print("  phonemes:", " ".join(labels[i].strip() for i in trial.target("phonemes")[:10]))
else:
    print("Session file not found. Run the export command above to create it.")


t12.2022.05.19: 200 trials


t12.2022.05.19/test/0000 · Yet we no longer feel uneasy.
  neural/spike_pow: (290, 256) float32
  neural/tx: (290, 256) uint8
  neural/tx2: (290, 256) uint8
  neural/tx3: (290, 256) uint8
  neural/tx4: (290, 256) uint8
  phonemes: Y EH T | W IY | N OW |


## Read the archived results

The release calls the evaluated partition test. This reproduction used its labeled trials for validation. Figure 2c uses the authors' online trials, and Figure 3a uses the offline validation archive.

This walkthrough uses the checked-in archive and takes no data download.

**Which trials are scored, and how to compare with the paper.** The paper's evaluation split is the one the release ships; this pipeline reuses it, so a fresh `trials.csv` covers the same trials as the archived score. The split columns are documented in [`data/splits/README.md`](data/splits/README.md), the paper-figure selection in the [evaluation protocol](docs/20261124_evaluation_protocol/README.md), and the fresh-vs-archived WER comparison recipe in the [fresh-machine guide](../../fresh_machine_setup.md) — match the score-bearing run length before comparing WERs with the paper.

In [2]:
import csv
import json
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "figures/code"))
from figure_paths import PUBLISHED, load_benchmark

benchmark = load_benchmark()

def read_rows(path):
    """Read the archived CSV rows in their original order."""
    with (root / path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

## From a trial or day to the reported result

In [3]:
arm = "Willett 2023 (T12)"
from build_session_per_vs_wer import build, read_tsv, word_ed

source = benchmark["figure3_per_sources"][arm]
rows = read_rows(source["greedy"])
decoded = read_tsv(root / source["decode"])
trial = rows[0]
reference, hypothesis = decoded[0]
print(f"{trial['session']} · {reference}")
print(f"Word hypothesis: {hypothesis}")
edits, words = word_ed(reference, hypothesis)
print(f"Trial word edits: {edits}/{words}")
result = build(arm, root / source["greedy"], root / source["decode"])
day = result["sessions"][0]
print(f"{day['session']}: {day['n']} trials, PER {day['per']:.2f}%, WER {day['wer']:.2f}%")
print(f"{len(result['sessions'])} sessions, {len(rows)} evaluated trials; pooled {result['pooled']}")

online = [r for r in read_rows(benchmark["panel_cd_daily_csv"]) if r["series"] == "willett23_t12"]
paired = [r for r in read_rows(PUBLISHED / "figure2cd_paired_trials.csv") if r["series"] == "willett23_t12"]
print(f"Figure 2c: {len(online)} days, {sum(int(r['wer_trials']) for r in online)} trials")
print(f"Figure 3a: {len(result['sessions'])} days, {len(rows)} trials")
print(f"Matched online and offline trials: {len(paired)}")
assert (len(online), sum(int(r["wer_trials"]) for r in online),
        len(result["sessions"]), len(rows), len(paired)) == (5, 400, 24, 880, 200)

t12.2022.04.28 · Theocracy reconsidered.
Word hypothesis: THE WEEKEND
Trial word edits: 2/2
Willett 2023 (T12): 24 sessions | pooled PER 23.42% | WER 23.26%
t12.2022.04.28: 20 trials, PER 41.28%, WER 59.06%
24 sessions, 880 evaluated trials; pooled {'per': 23.42, 'wer': 23.26}
Figure 2c: 5 days, 400 trials
Figure 3a: 24 days, 880 trials
Matched online and offline trials: 200


## Reproduce and score

[Common reader guide](../../docs/20261124_repository_migration/reader_workflow.md) · [Study evaluation protocol](docs/20261124_evaluation_protocol/README.md)

### Train and infer

Run in PowerShell at the repository root after preparing the trainer HDF5 files. The batch count below gives a short training check. The [code guide](code/README.md) identifies the longer score-bearing runs.

```powershell
$study = "studies/20230831_willett"
$date = Get-Date -Format yyyyMMdd
$run = "$study/results/${date}_dryad_x69p8czpq__hdf5__ctc_greedy__reader/runtime"
python "$study/code/train.py" 1000 "$run" 1000
python "$study/code/infer.py" "$run" "$run/t12_logits.hdf5"
```

### Calculate metrics

```powershell
$score = "$study/results/${date}_dryad_x69p8czpq__hdf5__ctc_greedy__reader_per"
python "$study/code/evaluate.py" "$run" --output-dir "$score"
```

Open `$score/summary.json` for pooled phoneme error rate (PER) and `trials.csv` for the trial counts. For fresh word error rate (WER), use [speech word decoding in WSL](../../docs/20261124_repository_migration/reader_workflow.md#speech-word-error-rate-in-wsl) with the new `t12_logits.hdf5` and the Willett 2023 row. The Notebook's archived WER uses an additional OPT rescoring stage.
